# CGHD -> YOLOv8: hand-drawn circuit symbol detector
Runtime -> Change runtime type -> **T4 GPU** before running.
Trains on the full CGHD dataset (3,269 photos, 59 labels merged to 15).
Expect ~75-90 min on a T4. Keep this tab open.

In [ ]:
# Cell 1 - setup
!nvidia-smi
%pip install -q ultralytics huggingface_hub
from google.colab import drive
drive.mount('/content/drive')   # so a disconnect does not lose the run

In [ ]:
# Cell 2 - download CGHD (annotations + images only; skips ~2 GB of masks/polygons)
from huggingface_hub import snapshot_download
ROOT = snapshot_download(
    repo_id="lowercaseonly/cghd", repo_type="dataset",
    local_dir="/content/cghd", max_workers=8,
    allow_patterns=["classes*.json", "drafter_*/annotations/*", "drafter_*/images/*"],
)
print(ROOT)
!du -sh /content/cghd

In [ ]:
# Cell 3 - sanity check the label taxonomy before converting anything
import json, collections, glob, xml.etree.ElementTree as ET
print(len(json.load(open(f"{ROOT}/classes.json"))), "classes in classes.json")
cnt = collections.Counter()
for x in glob.glob(f"{ROOT}/drafter_*/annotations/*.xml"):
    try:
        for o in ET.parse(x).getroot().iter("object"):
            cnt[o.findtext("name")] += 1
    except Exception:
        pass
print(f"{sum(cnt.values())} boxes across {len(cnt)} distinct labels")
for k, v in cnt.most_common():
    print(f"{v:>7}  {k}")

In [ ]:
# Cell 4 - PASCAL VOC -> YOLO, 59 labels merged to 15, split BY DRAFTER
import os, re, yaml, xml.etree.ElementTree as ET
from pathlib import Path

MERGE_RULES = [
    ("transistor.bjt","transistor_bjt"), ("transistor.fet","transistor_fet"),
    ("transistor.mos","transistor_fet"), ("transistor.photo","transistor_bjt"),
    ("transistor","transistor_bjt"), ("resistor","resistor"),
    ("capacitor","capacitor"), ("inductor","inductor"), ("diode","diode"),
    ("operational_amplifier","opamp"), ("voltage","source_voltage"),
    ("current","source_current"), ("battery","source_voltage"),
    ("gnd","gnd"), ("ground","gnd"), ("junction","junction"),
    ("crossover","crossover"), ("switch","switch"), ("terminal","terminal"),
    ("connector","terminal"), ("port","terminal"), ("text","text"),
]
NAMES = ["resistor","capacitor","inductor","diode","transistor_bjt",
         "transistor_fet","source_voltage","source_current","opamp","gnd",
         "junction","crossover","switch","terminal","text","other_component"]
IDX = {n: i for i, n in enumerate(NAMES)}

def merge(raw):
    n = (raw or "").strip().lower().replace(" ", "_").replace("-", "_")
    for p, m in MERGE_RULES:
        if n.startswith(p):
            return m
    return "other_component"

drafters = sorted((p for p in Path(ROOT).glob("drafter_*") if p.is_dir()),
                  key=lambda p: int(re.search(r"-?\d+", p.name).group()
                                    if re.search(r"-?\d+", p.name) else 0))
test_d, val_d = set(drafters[-4:]), set(drafters[-7:-4])
print("test:", [d.name for d in test_d])
print("val :", [d.name for d in val_d])
print("train:", len(drafters) - 7, "drafters")

OUT = Path("/content/yolo_cghd")
for s in ("train", "val", "test"):
    (OUT/"images"/s).mkdir(parents=True, exist_ok=True)
    (OUT/"labels"/s).mkdir(parents=True, exist_ok=True)

kept = dropped = 0
for d in drafters:
    split = "test" if d in test_d else "val" if d in val_d else "train"
    for xml in sorted((d/"annotations").glob("*.xml")):
        img = None
        for e in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
            p = d/"images"/(xml.stem + e)
            if p.exists():
                img = p; break
        if img is None:
            continue
        try:
            r = ET.parse(xml).getroot()
        except Exception:
            continue
        sz = r.find("size")
        if sz is None:
            continue
        W, H = int(float(sz.findtext("width") or 0)), int(float(sz.findtext("height") or 0))
        if not W or not H:
            continue
        lines = []
        for o in r.iter("object"):
            m, bb = merge(o.findtext("name")), o.find("bndbox")
            if m is None or bb is None:
                dropped += 1; continue
            try:
                x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin","ymin","xmax","ymax"))
            except (TypeError, ValueError):
                dropped += 1; continue
            x1, x2 = sorted((max(0,min(x1,W)), max(0,min(x2,W))))
            y1, y2 = sorted((max(0,min(y1,H)), max(0,min(y2,H))))
            if x2-x1 < 2 or y2-y1 < 2:
                dropped += 1; continue
            lines.append(f"{IDX[m]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} "
                         f"{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
            kept += 1
        if not lines:
            continue
        stem = f"{d.name}_{xml.stem}"
        dst = OUT/"images"/split/(stem + img.suffix)
        if not dst.exists():
            os.symlink(img, dst)
        (OUT/"labels"/split/(stem + ".txt")).write_text(chr(10).join(lines))

yaml.safe_dump({"path": str(OUT), "train": "images/train", "val": "images/val",
                "test": "images/test", "names": {i: n for i, n in enumerate(NAMES)}},
               open(OUT/"data.yaml", "w"), sort_keys=False)
print()
print(f"kept {kept} boxes, dropped {dropped} ({100*dropped/(kept+dropped):.1f}%)")
for s in ("train","val","test"):
    print(s, len(list((OUT/"images"/s).iterdir())), "images")

In [ ]:
# Cell 5 - train
from ultralytics import YOLO
m = YOLO("yolov8s.pt")
m.train(data="/content/yolo_cghd/data.yaml", imgsz=896, epochs=50, batch=-1,
        patience=12, degrees=10, fliplr=0.5, flipud=0.3, scale=0.5, mosaic=1.0,
        workers=2, cache=False,
        project="/content/drive/MyDrive/mlee_cghd", name="yolov8s")
print(m.val(split="test"))

In [ ]:
# Cell 6 - zip the run and download best.pt
!cd /content/drive/MyDrive/mlee_cghd && zip -r /content/cghd_run.zip yolov8s -x "*/weights/last.pt"
from google.colab import files
files.download("/content/cghd_run.zip")